In [1]:
pip install requests

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [1]:
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("CH_API_KEY")
if not API_KEY:
    raise RuntimeError("CH_API_KEY not found. Did you create a .env file?")

# ==============================
# 0. CONFIGURATION
# ==============================
BASE_URL = "https://api.company-information.service.gov.uk"

# Example SIC cluster for Week 1: B2B / consulting / advertising
sic_cluster_b2b_services = [70229, 70221, 73110]


# ==============================
# 1. LOW-LEVEL HTTP HELPER
# ==============================

def make_request(endpoint: str, params: dict | None = None) -> dict:
    """
    Call Companies House API at BASE_URL + endpoint with optional query params.
    Returns the JSON response as a Python dict or raises HTTPError on failure.
    """
    url = f"{BASE_URL}{endpoint}"
    resp = requests.get(
        url,
        auth=(API_KEY, ""),              # API key as username, blank password
        headers={"Accept": "application/json"},
        params=params,
        timeout=10,
    )
    resp.raise_for_status()
    return resp.json()


# ==============================
# 2. COMPANY-SPECIFIC HELPERS
# ==============================

def fetch_company_profile(company_number: str) -> dict:
    """Get the basic company profile."""
    return make_request(f"/company/{company_number}")


def fetch_filing_history(company_number: str, items_per_page: int = 200) -> dict:
    """Get filing history (list of submitted documents) for a company."""
    params = {"items_per_page": items_per_page, "start_index": 0}
    return make_request(f"/company/{company_number}/filing-history", params=params)


def fetch_charges(company_number: str, items_per_page: int = 200) -> dict:
    """Get list of registered charges (secured lending) for a company."""
    params = {"items_per_page": items_per_page, "start_index": 0}
    return make_request(f"/company/{company_number}/charges", params=params)


# ==============================
# 3. ADVANCED SEARCH (BUILD UNIVERSE)
# ==============================

def advanced_search_companies(
    sic_codes,
    company_status: str = "active",
    size: int = 100,
    start_index: int = 0,
    location: str | None = None,
) -> dict:
    """
    Call /advanced-search/companies filtered by SIC codes (and optionally status, location).
    Returns the JSON response as a Python dict.
    """
    # Convert list of SIC codes -> "70229,70221,73110"
    if isinstance(sic_codes, (list, tuple, set)):
        sic_param = ",".join(str(code) for code in sic_codes)
    else:
        sic_param = str(sic_codes)

    params = {
        "sic_codes": sic_param,
        "company_status": company_status,  # e.g. "active"
        "size": size,                      # how many results to return (max 5000)
        "start_index": start_index,        # paging, start at 0 for now
    }

    if location:
        params["location"] = location  # e.g. "london"

    # Reuse our generic helper
    return make_request("/advanced-search/companies", params=params)


def companies_json_to_df(result_dict: dict) -> pd.DataFrame:
    """
    Convert advanced-search JSON result into a pandas DataFrame
    with the columns we care about.
    Columns: company_number, name, sic_codes, registered_office_address.
    """
    items = result_dict.get("items", [])
    rows = []

    for item in items:
        roa = item.get("registered_office_address") or {}

        # Build a single address string from the parts
        address_parts = [
            roa.get("address_line_1"),
            roa.get("address_line_2"),
            roa.get("locality"),
            roa.get("region"),
            roa.get("postal_code"),
            roa.get("country"),
        ]
        # Keep only non-empty parts and join with ", "
        address = ", ".join(part for part in address_parts if part)

        rows.append(
            {
                "company_number": item.get("company_number"),
                "name": item.get("company_name"),
                "sic_codes": ";".join(item.get("sic_codes") or []),
                "registered_office_address": address,
            }
        )

    return pd.DataFrame(rows)


def build_seed_universe_if_needed(
    sic_codes,
    company_status: str = "active",
    size: int = 100,
    csv_path: Path = Path("data/companies_seed.csv"),
) -> pd.DataFrame:
    """
    If data/companies_seed.csv exists, load and return it.
    Otherwise, call advanced_search_companies to build a seed universe,
    save it to CSV, and return the DataFrame.
    """
    if csv_path.exists():
        print(f"Found existing seed file at {csv_path}, loading it...")
        return pd.read_csv(csv_path)

    print("No seed file found. Calling Companies House API to build one...")

    # 1) Call advanced search
    result = advanced_search_companies(
        sic_codes=sic_codes,
        company_status=company_status,
        size=size,
        start_index=0,
        location=None,  # optionally restrict location here
    )

    print("Total matches reported by API (hits):", result.get("hits"))
    print("Items in this page:", len(result.get("items", [])))

    # 2) Convert to DataFrame
    seed_df = companies_json_to_df(result)

    # 3) Save to CSV
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    seed_df.to_csv(csv_path, index=False)
    print(f"Saved {len(seed_df)} companies to {csv_path}")

    return seed_df


# ==============================
# 4. VERY SIMPLE EXPLORATION
# ==============================

def explore_sample(seed_df: pd.DataFrame, n_sample: int = 10) -> None:
    """
    For n_sample random companies from seed_df:
    - print company number and name
    - print status
    - print number of filings and charges
    """
    n = min(n_sample, len(seed_df))
    sample_df = seed_df.sample(n=n, random_state=42)

    for _, row in sample_df.iterrows():
        company_number = str(row["company_number"])
        company_name = row["name"]

        print("=" * 80)
        print(f"{company_number} – {company_name}")

        try:
            # --- Call the three endpoints ---
            profile = fetch_company_profile(company_number)
            filings = fetch_filing_history(company_number)
            charges = fetch_charges(company_number)

            # --- Extract what we care about ---
            status = profile.get("company_status")
            filings_count = filings.get("total_count", len(filings.get("items", [])))
            charges_count = charges.get("total_count", len(charges.get("items", [])))

            print(f"Status        : {status}")
            print(f"# of filings  : {filings_count}")
            print(f"# of charges  : {charges_count}")

        except Exception as e:
            print("Error fetching data:", e)


def quick_universe_summary(seed_df: pd.DataFrame, n_sample: int = 20) -> pd.DataFrame:
    """
    Take a random sample of companies and build a small summary DataFrame with:
    - company_number
    - status
    - filings_count
    - charges_count
    """
    records = []
    n = min(n_sample, len(seed_df))

    for _, row in seed_df.sample(n=n, random_state=0).iterrows():
        cn = str(row["company_number"])

        try:
            profile = fetch_company_profile(cn)
            filings = fetch_filing_history(cn)
            charges = fetch_charges(cn)

            records.append(
                {
                    "company_number": cn,
                    "status": profile.get("company_status"),
                    "filings_count": filings.get(
                        "total_count", len(filings.get("items", []))
                    ),
                    "charges_count": charges.get(
                        "total_count", len(charges.get("items", []))
                    ),
                }
            )
        except Exception as e:
            print(f"Error fetching data for {cn}: {e}")

    return pd.DataFrame(records)


# ==============================
# 5. MAIN EXECUTION (RUN IN NOTEBOOK)
# ==============================

# Build or load the seed universe for your B2B services cluster
seed_df = build_seed_universe_if_needed(
    sic_codes=sic_cluster_b2b_services,
    company_status="active",
    size=100,
)

print("\n=== Simple exploration for 10 random companies ===")
explore_sample(seed_df, n_sample=10)

print("\n=== Quick summary for 20 companies ===")
summary_df = quick_universe_summary(seed_df, n_sample=20)
display(summary_df)

print("\nStatus counts:")
print(summary_df["status"].value_counts())

print("\nCharges count distribution (top):")
print(summary_df["charges_count"].value_counts().head())

print("\nFilings count summary:")
print(summary_df["filings_count"].describe())


Found existing seed file at data\companies_seed.csv, loading it...

=== Simple exploration for 10 random companies ===
12846804 – AIO APP LIMITED
Status        : active
# of filings  : 16
# of charges  : 0
08800582 – RADHA KRISHNA AGS LTD
Status        : active
# of filings  : 37
# of charges  : 0
11985944 – MY TRAINING RESOURCES LTD
Status        : active
# of filings  : 18
# of charges  : 0
11686882 – JIMMY JAMES COMPANIES UK LTD
Status        : active
# of filings  : 25
# of charges  : 0
11337773 – SHAPE YOUR FUTURE ONLINE LTD
Status        : active
# of filings  : 21
# of charges  : 0
11504523 – DOCTORS IN BUSINESS LTD
Status        : active
# of filings  : 18
# of charges  : 0
10266613 – MESOLEAN GROUP LTD.
Status        : active
# of filings  : 20
# of charges  : 0
15054797 – BAHELMI CONSULTANCY LTD
Status        : active
# of filings  : 7
# of charges  : 0
11797482 – CARNE CONSULTING LIMITED
Status        : active
# of filings  : 8
# of charges  : 0
NI050108 – MCMILLEN CONSULTAN

,company_number,status,filings_count,charges_count
0,12423136,active,14,0
1,11816651,active,20,0
2,12549088,active,17,0
3,SC413867,active,32,0
4,12434325,active,18,0
5,11787455,active,22,0
6,10780140,active,15,0
7,11923073,active,24,0
8,14087296,active,7,0
9,03369943,active,74,2



Status counts:
active    20
Name: status, dtype: int64

Charges count distribution (top):
0    17
2     2
1     1
Name: charges_count, dtype: int64

Filings count summary:
count    20.000000
mean     29.900000
std      23.536981
min       7.000000
25%      16.500000
50%      21.500000
75%      33.250000
max      83.000000
Name: filings_count, dtype: float64
